In [1]:
!pip install pandas
!pip install openpyxl

In [4]:
import pandas as pd

# STEP 1: Load data from 'AllData' sheet
df = pd.read_excel(
    r'C:\Users\dolla\Desktop\Data Analyst Projects\Air Quality\2024_All_sites_air_quality_hourly_avg_AIR-I-F-V-VH-O-S1-DB-M2-4-0.xlsx',
    sheet_name="AllData",
    parse_dates=["datetime_AEST"]
)

# STEP 2: Check Basic Structure
print("Initial shape:", df.shape)
print("Columns:\n", df.columns)
print("Data types:\n", df.dtypes)

Initial shape: (816133, 14)
Columns:
 Index(['datetime_AEST', 'datetime_local', 'location_id', 'location_name',
       'latitude', 'longitude', 'value', 'validation_flag', 'parameter_name',
       'parameter_method_name', 'parameter_description',
       'analysis_method_name', 'unit_of_measure', 'method_quality'],
      dtype='object')
Data types:
 datetime_AEST            datetime64[ns]
datetime_local                   object
location_id                       int64
location_name                    object
latitude                        float64
longitude                       float64
value                           float64
validation_flag                  object
parameter_name                   object
parameter_method_name            object
parameter_description            object
analysis_method_name             object
unit_of_measure                  object
method_quality                   object
dtype: object


In [5]:
# STEP 3: Check for missing values per column
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))


Missing values per column:
datetime_AEST            0
datetime_local           0
location_id              0
location_name            0
latitude                 0
longitude                0
value                    0
validation_flag          0
parameter_name           0
parameter_method_name    0
parameter_description    0
analysis_method_name     0
unit_of_measure          0
method_quality           0
dtype: int64


No missing value. Thus, step 4 is not necessary. I put step 4 in there for learning purpose.

In [6]:
# STEP 4: Drop rows with missing critical info: datetime, value, location_name, parameter_name
#df = df.dropna(subset=["datetime_AEST", "value", "location_name", "parameter_name"])
#print("Shape after dropping rows with missing critical info:", df.shape)

In [7]:
# STEP 5: Find the duplicates rows
duplicates = df[df.duplicated()]
print(f"Number of duplicate rows: {duplicates.shape[0]}")
duplicates.head()

Number of duplicate rows: 38181


,datetime_AEST,datetime_local,location_id,location_name,latitude,longitude,value,validation_flag,parameter_name,parameter_method_name,parameter_description,analysis_method_name,unit_of_measure,method_quality
47664,2024-01-24 12:00:00,2024-01-24 13:00:00,1112,Brooklyn,-37.822098,144.8471,22.281,Y,BSP,BSP,Scattering coefficent of light due to particles,Light scattering method - Nephelometer,1/Mm,Equivalence Method
47680,2024-01-24 12:00:00,2024-01-24 13:00:00,1112,Brooklyn,-37.822098,144.8471,22.281,Y,BSP,BSP,Scattering coefficent of light due to particles,Light scattering method - Nephelometer,1/Mm,Equivalence Method
47750,2024-01-24 13:00:00,2024-01-24 14:00:00,1112,Brooklyn,-37.822098,144.8471,38.667,Y,BSP,BSP,Scattering coefficent of light due to particles,Light scattering method - Nephelometer,1/Mm,Equivalence Method
47766,2024-01-24 13:00:00,2024-01-24 14:00:00,1112,Brooklyn,-37.822098,144.8471,38.667,Y,BSP,BSP,Scattering coefficent of light due to particles,Light scattering method - Nephelometer,1/Mm,Equivalence Method
47836,2024-01-24 14:00:00,2024-01-24 15:00:00,1112,Brooklyn,-37.822098,144.8471,42.694,Y,BSP,BSP,Scattering coefficent of light due to particles,Light scattering method - Nephelometer,1/Mm,Equivalence Method


In [15]:
# STEP 6: Drop duplicates
df = df.drop_duplicates()

In [18]:
# STEP 7: Recheck for duplicates in the updated dataframe
duplicates = df[df.duplicated()]
print(f"Number of duplicate rows after cleaning: {duplicates.shape[0]}")

Number of duplicate rows after cleaning: 0


In [19]:
# STEP 8: Clean text fields
df["location_name"] = df["location_name"].str.strip()
df["parameter_name"] = df["parameter_name"].str.strip()

In [20]:
# STEP 9: Add date/time features
df["date"] = df["datetime_AEST"].dt.date
df["hour"] = df["datetime_AEST"].dt.hour
df["month"] = df["datetime_AEST"].dt.month
df["weekday"] = df["datetime_AEST"].dt.day_name()


In [22]:
# STEP 10: Add season column
season_map = {
    12: "Summer", 1: "Summer", 2: "Summer",
    3: "Autumn", 4: "Autumn", 5: "Autumn",
    6: "Winter", 7: "Winter", 8: "Winter",
    9: "Spring", 10: "Spring", 11: "Spring"
}
df["season"] = df["month"].map(season_map)

In [24]:
# STEP 11: Handle invalid numeric ranges based on parameter type

# Parameters that must be non-negative
must_be_non_negative = ["SWS", "VWS", "SIG60"]
df.loc[(df["parameter_name"].isin(must_be_non_negative)) & (df["value"] < 0), "value"] = None

# Parameters that must be between 0 and 360
must_be_0_to_360 = ["SWD", "VWD"]
df.loc[(df["parameter_name"].isin(must_be_0_to_360)) & ((df["value"] < 0) | (df["value"] > 360)), "value"] = None

In [25]:
# STEP 12: Handle extreme outliers ONLY for pollutant parameters
pollutants = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "BSP"]

for param in pollutants:
    upper = df[df["parameter_name"] == param]["value"].quantile(0.999)
    df.loc[df["parameter_name"] == param, "value"] = df.loc[df["parameter_name"] == param, "value"].clip(upper=upper)

In [26]:
# STEP 13: Export cleaned data to CSV for dashboard use
df.to_csv("clean_air_quality_data.csv", index=False)
print("✅ Data cleaning complete. File saved as 'clean_air_quality_data.csv'")

✅ Data cleaning complete. File saved as 'clean_air_quality_data.csv'
